Grupo 4 - 13/05/2026 - IA - Trabajo IA - Motor de Búsqueda VP-Tree

# Notebook 4: Motor de Búsqueda con VP-Tree y Demo

Construcción del índice de búsqueda facial mediante VP-Tree, pipeline completo de consulta y demostración interactiva con Gradio desplegada en Hugging Face Spaces.

## 0. Instalación

In [25]:
%%capture
!pip install vptree deepface tf-keras gradio
!pip install scipy -q

In [26]:
import pickle
import time
import numpy as np
import vptree
from pathlib import Path
from deepface import DeepFace
from scipy.spatial import KDTree

## 1. Carga del Dataset de Caras y embeddings

In [3]:
!wget -q EMBEDDINGS_URL

with open('embeddings.pkl', 'rb') as f:
    embeddings = pickle.load(f)

print(f"[INFO] Embeddings cargados : {len(embeddings)}")
print(f"[INFO] Dimensión           : {len(next(iter(embeddings.values())))}")

[INFO] Embeddings cargados : 38299
[INFO] Dimensión           : 128


In [4]:
%%capture
!wget FACE_CROPS_URL -O carasRecortadas.zip
!unzip carasRecortadas.zip -d carasRecortadas

## 2. Construcción del VP-Tree

Construimos el VP-Tree siguiendo el patrón de [PyImageSearch](https://pyimagesearch.com/2019/08/26/building-an-image-hashing-search-engine-with-vp-trees-and-opencv/). La diferencia respecto al artículo es que en lugar de distancia Hamming entre hashes enteros usamos distancia euclídea entre embeddings de 128 dimensiones, que es la métrica con la que Facenet fue entrenado.

La complejidad de construcción es O(n log n) y la de consulta O(log n), frente al O(n) de una búsqueda lineal.

In [28]:
paths    = list(embeddings.keys())
vecs_arr = np.array([embeddings[k] for k in paths], dtype=np.float32)

In [29]:
def distancia(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

print("[INFO] building KDTree...")
t0   = time.time()
tree = KDTree(vecs_arr)
print(f"[INFO] KDTree construido en {time.time() - t0:.1f}s")

[INFO] building KDTree...
[INFO] KDTree construido en 0.1s


## 3. Pipeline de búsqueda

La función `buscar` recibe la ruta de una imagen de consulta y devuelve los `top_k` recortes más similares del dataset con su distancia euclídea. Internamente sigue el mismo flujo que el Notebook 2: extrae el embedding con Facenet y consulta el VP-Tree con `get_n_nearest_neighbors`.

In [30]:
def buscar(img_path, top_k=5):
    emb = DeepFace.represent(img_path=str(img_path),model_name='Facenet',enforce_detection=False)[0]['embedding']
    query_vec = np.array(emb, dtype=np.float32)
    dists, idxs = tree.query(query_vec, k=top_k)
    return [(float(d), paths[i]) for d, i in zip(dists, idxs)]

### 3.1 Prueba de búsqueda

Verificamos el pipeline sobre una cara aleatoria del propio dataset. La primera coincidencia debe ser la imagen de consulta con distancia ≈ 0.

In [31]:
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [32]:
carasPath = Path('carasRecortadas')
caras = list(carasPath.rglob('*.jpg'))
print(f"Caras disponibles localmente: {len(caras)}")

Caras disponibles localmente: 38299


Vamos a probar que el motor de busqueda esta funcionando correctamente con una muestra dfe 5 caras aleatorias.

In [33]:
random.seed(42)
consultas = random.sample(caras, 5)

In [34]:
for idx, query_path in enumerate(consultas):
    results = buscar(query_path, top_k=5)

    fig, axes = plt.subplots(1, len(results) + 1, figsize=(3 * (len(results) + 1), 3))

    axes[0].imshow(mpimg.imread(query_path))
    axes[0].set_title('Consulta', fontsize=9)
    axes[0].axis('off')

    for ax, (dist, nombre) in zip(axes[1:], results):
        ruta = carasPath / nombre
        if ruta.exists():
            ax.imshow(mpimg.imread(ruta))
        ax.set_title(f'd={dist:.2f}', fontsize=9)
        ax.axis('off')

    plt.suptitle(f'Prueba {idx + 1} — {query_path.name}', fontsize=11)
    plt.tight_layout()
    plt.show()

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

<Figure size 1800x300 with 6 Axes>

## 4. Demo con Gradio

Desplegamos el  motor de busqueda faciales en  HuggingFace Spaces. A diferencia de de los